# 🧪 Ambiente Interativo: Detecção de Objetos

Neste notebook você testará a utilização prática do pipeline da biblioteca **Hugging Face (`transformers`)** para realizar inferências de "Object Detection".
Vamos garantir que seu ambiente utilize o PyTorch atrelado à placa de vídeo (Aceleração por CUDA).

In [ ]:
import torch

# Teste de Capacidade Térmica / Hardware
has_cuda = torch.cuda.is_available()
print("Ambiente possui núcleos CUDA ativos?", has_cuda)

if has_cuda:
    print("✅ Placa de Vídeo Detectada:", torch.cuda.get_device_name(0))
else:
    print("❌ Atenção: Nenhuma aceleração CUDA detectada. O modelo pode demorar mais executando em CPU. Instale o PyTorch apropriado.")

In [ ]:
from transformers import pipeline

# Mapearemos `device=0` o que força o framework a utilizar a GPU Principal do seu PC (Cuda 0)
hw_device = 0 if has_cuda else -1

print("Aguarde. Carregando modelo `facebook/detr-resnet-50` para memória...")
pipe = pipeline("object-detection", model="facebook/detr-resnet-50", device=hw_device)
print("Modelo Carregado com sucesso e pronto para inferências em menos de 1 segundo!")

In [ ]:
import urllib.request
from PIL import Image, ImageDraw

# Etapa 1: Carregar uma imagem. Podemos gerar um download rápido de exemplo das ruas.
url = "https://images.unsplash.com/photo-1543852786-1cf6624b9987?ixlib=rb-4.0.3&q=80"
img_path = "imagem_teste.jpg"
urllib.request.urlretrieve(url, img_path)

imagem_cv = Image.open(img_path)
print("Analise visual da foto de origem iniciada.")

# Etapa 2: Pedir via variável 'pipe' que ele detecte os objetos
resultados = pipe(imagem_cv)
print(f"\nEncontrados {len(resultados)} objetos. Estrutura Padrão:")
print(resultados)

# Etapa 3: Transformar as coordenadas de volta para desenhos (Caixas vermelhas)
desenhista = ImageDraw.Draw(imagem_cv)
for obj in resultados:
    box = obj['box']
    # Limiar: Se tiver muito em dúvida, não plotamos na tela.
    if obj['score'] < 0.60:
        continue
        
    # Desenha retângulo
    desenhista.rectangle(
        [(box['xmin'], box['ymin']), (box['xmax'], box['ymax'])], 
        outline="red", width=3
    )
    # Desenha o texto informando a Confiança (Accuracy)
    label = f"{obj['label'].upper()} ({obj['score']*100:.1f}%)"
    desenhista.text((box['xmin'], max(0, box['ymin']-15)), label, fill="red")

# Aqui apenas exibimos o resultado da foto para você:
imagem_cv